In [ ]:
# Core Algorithms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Preprocessing and Data Splitting
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
from sklearn import set_config;set_config(transform_output = 'default')

# Learning Models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

""" These models are not needed suitable for this
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
"""
# Boosting 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import joblib

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"C:\Users\sakth\Anaconda\Diabetes_ete\diabetes_prediction_dataset.csv")

In [ ]:
# Cheking for missing values
print (df.info())
print ("\n")
print ("No.of Null values : \n",df.isnull().sum())
print ("No.of duplicated values : ", df.duplicated().sum())
# Removing missing values
df=df.drop_duplicates()
print ("No.of duplicated values after cleaning: ", df.duplicated().sum())

In [ ]:
# Mapping Smoking History Column
mapping = {"never": "No" ,"No Info":"Not willing to disclose","former":"Quitted","current":"Yes","not current":"Quitted","ever":"No"}
df['smoking_history'] = df['smoking_history'].map(mapping)

In [ ]:
# Data Splitting
X = df.drop('diabetes',axis =1)
y = df['diabetes']

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state =42, stratify = y)

#Column Transform
num_col = X.select_dtypes(include = ['float64','int64']).columns
cat_col = X.select_dtypes(include = ['object']).columns

preprocessor = ColumnTransformer([
    ('num',StandardScaler(),num_col),
    ('cat',OneHotEncoder(handle_unknown = 'ignore'),cat_col)
])

In [ ]:
# Model Defining

models = {
    'Log_Reg' : {
        'model' : LogisticRegression(max_iter = 500),
        'params' : {'model__C' : [0.1,1,10] }
    },
    
    'RandomForest' : {
        'model' : RandomForestClassifier(),
        'params' : {
            'model__n_estimators' : [100,200],
            'model__max_depth' : [5,10]
        }
    },
    'GradientBoost' : {
        'model' : GradientBoostingClassifier(),
        'params' : {
            'model__n_estimators' : [100,200],
            'model__learning_rate' : [0.01,0.1]
        }
    },
    'XGBoost' : {
        'model' : XGBClassifier(eval_metric = 'logloss'),
        'params' : {
            'model__n_estimators' : [100,200],
            'model__learning_rate' : [0.05,0.1],
            'model__max_depth' : [3,6]
        }
    },
    'LightGBM' : {
        'model' : LGBMClassifier(class_weight='balanced', verbose=-1),
        'params' : {
            'model__n_estimators' : [100,200],
            'model__learning_rate' : [0.05,0.1]
        }
    }
}
    """ 
    These models are not needed suitable for this
    'SVM' : {
        'model' : SVC(probability = True),
        'params' : {
            'model__C' : [0.1,1],
            'model__kernel' : ['linear','rbf']
        }
    },
    'KNN' : {
        'model' : KNeighborsClassifier(),
        'params' : {
            'model__n_neighbors' : [3,5,7] 
        }
    },
    'Naive_Bayes' :{
        'model' : GaussianNB(),
        'params' : {}
    },
    'DecisionTree' : {
        'model' : DecisionTreeClassifier(),
        'params' : {
            'model__max_depth' : [5,10,20],
            'model__min_samples_split' : [2,5]
        }
    }
    """

In [ ]:
# Model Training with parameter tuning

best_models = {}
results = []

for name, config in models.items():
    print(f'***** Training {name} *****')
    pipe = Pipeline([
        ('preprocessor',preprocessor),
        ('model',config['model'])
    ])
    grid = GridSearchCV(pipe, config['params'], cv=5, scoring = 'f1', n_jobs =1 )
    grid.fit(X_train,y_train)
    best_model = grid.best_estimator_
    best_models[name] = best_model
    y_pred = best_model.predict(X_test)
    results.append({
        'model' : name,
        'accuracy' : accuracy_score(y_test,y_pred),
        'precision' : precision_score(y_test,y_pred),
        'recall' : recall_score(y_test,y_pred),
        'f1' : f1_score(y_test,y_pred),
        'roc_auc' : roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1])
    })


In [ ]:
result_df = pd.DataFrame(results)
print(result_df)

In [ ]:
best_model_name_JN = 
best_model_JN = best_models[best_model_name_JN]
print (" Best Model : ", best_model_JN)


In [ ]:
joblib.dump(best_model, "models/best_model.pkl")